## 01 — Load and Clean ANES Data

In [ ]:
import pandas as pd
import numpy as np

# set path to raw ANES CSV
DATA_PATH = "../data/anes_timeseries_cdf_csv_20260205.csv"

cols = ['VCF0004', 'VCF0830', 'VCF0114', 'VCF0301', 'VCF0886']
df_raw = pd.read_csv(DATA_PATH, usecols=cols, low_memory=False)
df_raw.shape

In [ ]:
def to_num(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors='coerce')

for col in cols:
    df_raw[col] = to_num(df_raw[col])

In [ ]:
df = df_raw.copy()

df['year']    = df['VCF0004'].where(df['VCF0004'] > 0)
df['redist']  = (8 - df['VCF0830']).where(df['VCF0830'].between(1, 7))
df['income']  = df['VCF0114'].where(df['VCF0114'].between(1, 5))
df['partyid'] = df['VCF0301'].where(df['VCF0301'].between(1, 7))
df['redist2'] = (8 - df['VCF0886']).where(df['VCF0886'].between(1, 7))

df['party3'] = pd.cut(
    df['partyid'],
    bins=[0, 2, 4, 7],
    labels=['Democrat', 'Independent', 'Republican']
)
df['lowinc'] = df['income'] <= 2

In [ ]:
clean = df[['year','redist','redist2','income','partyid','party3','lowinc']].dropna(subset=['year','redist','income','partyid'])
clean = clean[clean['year'] >= 1970]
print(f"Clean N: {len(clean):,}")
print(f"Years: {sorted(clean['year'].unique())}")

In [ ]:
clean.to_csv("../output/anes_clean.csv", index=False)
print("saved")